# 01 데이터 · 전처리 · 베이스라인

궁·능 23개소 일별 관람객. 이 노트북: 검산 → 전처리 상태 → 무료 개방 규칙 → 베이스라인 3개 + 서비스 수준 보정 → 자주 하는 실수 1·5 측정.

결정 근거는 `report/report.md` 부록 A11(의사결정 기록).

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager

# 묶음 폴더(real_final 또는 final) 찾기 — notebooks/ 에서 열든, 묶음 폴더나 그 위(Colab 의 /content)에서 열든 된다
HERE = Path.cwd()
CANDIDATES = [p / sub for p in [HERE, *HERE.parents] for sub in ('data', 'real_final/data', 'final/data')]
DATA = next((c for c in CANDIDATES if (c / 'metrics.csv').exists()), None)
if DATA is None:
    raise FileNotFoundError(f'data/metrics.csv 를 찾지 못했습니다 (지금 위치: {HERE}). '
                            '묶음 폴더 전체(data · notebooks ...)를 받았는지, 노트북을 notebooks/ 에서 열었는지 확인하세요.')
BASE = DATA.parent                              # 묶음 폴더
# 01 노트북만 쓰는 src/ 와 원본 데이터: 묶음 폴더 안(real_final: src/ · raw/)이 먼저, 없으면 저장소 최상위
SRC = next((p for p in [BASE / 'src', BASE.parent / 'src'] if (p / 'hv.py').exists()), None)
RAW = next((p for p in [BASE / 'raw', BASE.parent] if (p / 'anchors.json').exists()), None)
if SRC is not None:
    sys.path.insert(0, str(SRC))
print('데이터:', DATA)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 80)
HAVE = {f.name for f in font_manager.fontManager.ttflist}
KFONT = next((f for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR'] if f in HAVE), None)
if KFONT is None:
    print('※ 한글 폰트가 없어 그림의 한글이 네모로 보일 수 있습니다 (Colab: !apt -qq install fonts-nanum 후 런타임 다시 시작)')
plt.rcParams.update({'font.family': KFONT or 'sans-serif', 'axes.unicode_minus': False, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
SITES = ['경복궁', '창덕궁', '창경궁', '덕수궁', '종묘', '동구릉', '광릉', '홍유릉', '사릉', '태릉', '정릉', '헌릉', '선릉', '의릉',
         '영휘원', '서오릉', '서삼릉', '융릉', '파주삼릉', '파주장릉', '김포장릉', '양주온릉', '세종대왕유적']   # 궁 → 종묘 → 왕릉
BLUE, ORANGE, AQUA, GRAY, VIOLET = '#2a78d6', '#eb6834', '#1baf7a', '#898781', '#6250d6'

M = pd.read_csv(DATA / 'metrics.csv')
ORDER = M.drop_duplicates('방식').sort_values('방식 순서')['방식'].tolist()

def table(axis, metric, conf='전날', models=None, scale=1):
    """축(axis)의 값별로 방식 × 값 표. scale=1e8 이면 억 원."""
    x = M[(M['축'] == axis) & (M['확정'] == conf)]
    t = x.pivot(index='방식', columns='값', values=metric) / scale
    idx = [m for m in ORDER if m in t.index and (models is None or m in models)]
    return t.reindex(idx)

KEY = ['1 직전 7일 평균', '2 + 서비스 수준 보정', '7 MLP + 마지막 수요일 (FVA 2차)', 'C3 Chronos-2 + 공휴일·무료 개방']

데이터: c:\Users\tjwls\Downloads\real_final\real_final\data


## 1. 검산값 7개 (`anchors.json`)

**이 노트북만** 원본 데이터(`raw/`)와 `src/` 가 필요하다(real_final 에 들어 있음). 패키지: `pip install pandas pyarrow matplotlib "holidays==0.103"`. 02~05 는 `data/` 만 있으면 된다.

In [2]:
if SRC is None or RAW is None:
    raise FileNotFoundError(f'01 노트북은 src/hv.py 와 원본 데이터(raw/ 의 heritage_visitors_daily.parquet · anchors.json 등)가 필요합니다 '
                            f'(찾은 위치 src={SRC}, raw={RAW}). 02~05 노트북은 data/ 만으로 실행됩니다.')
try:
    import holidays
except ImportError:
    raise ImportError('01 노트북은 holidays 패키지가 필요합니다: pip install "holidays==0.103" 후 커널 다시 시작') from None
if holidays.__version__ != '0.103':
    print(f'※ holidays {holidays.__version__} — 분석은 0.103 으로 했습니다. 2026 공휴일 표시가 다를 수 있습니다.')
import hv
v, w, c, s = hv.load(RAW)
chk = hv.check_anchors(v, w, c, data_dir=RAW)
assert chk['일치'].all()
chk

※ holidays 0.105 — 분석은 0.103 으로 했습니다. 2026 공휴일 표시가 다를 수 있습니다.


,항목,읽은 값,anchors,일치
0,관람객 표 행 수,62028,62028,True
1,개소 수,23,23,True
2,total 합,119991391,119991391,True
3,외국인 4열 합,22960148,22960148,True
4,total 이 0 인 행,8832,8832,True
5,날씨 표 행 수,92414,92414,True
6,달력 행 수,4018,4018,True


## 2. 전처리 결과 — 상태 분류 (2023년 이후)

휴관·결측은 `y = NaN` 으로 학습·평가에서 뺀다. 규칙 밖의 0(대체 휴관·임시 휴관)도 뺀다(**A안**).

In [3]:
cal = hv.add_free_rule(hv.make_calendar())
panel = hv.build_panel(v, cal, s)
p23 = panel[panel['date'] >= '2023-01-01']
print(p23['status'].value_counts().to_string())
print('\n개소 수:', p23['site'].nunique(), '| 기간:', p23['date'].min().date(), '~', p23['date'].max().date())

status
open      21508
closed     3700

개소 수: 23 | 기간: 2023-01-01 ~ 2025-12-31


## 3. 무료 개방 규칙 (`is_free_rule` = 명절 연휴 + 매월 마지막 수요일, 2023년 이후)

In [4]:
fr = cal[cal['date'] >= '2023-01-01'].assign(연=lambda d: d['date'].dt.year)
fr.groupby('연')[['is_myeongjeol', 'is_last_wednesday', 'is_free_rule']].sum().rename(
    columns={'is_myeongjeol': '명절 연휴 일수', 'is_last_wednesday': '마지막 수요일', 'is_free_rule': '무료 개방일 합'})

,명절 연휴 일수,마지막 수요일,무료 개방일 합
연,,,
2023,10,12,22
2024,9,12,21
2025,13,12,24
2026,9,12,21


## 4. 베이스라인 3개 + 서비스 수준 보정 (원점 2025-01-01 부터 52개, 원점 전날 확정)

보정은 예측값을 바꾸지 않고 인력만 늘린다 → WAPE·MASE 는 같고 적중률·비용만 바뀐다.

In [5]:
base = ['0a 나이브', '0b 계절 나이브', '1 직전 7일 평균', '2 + 서비스 수준 보정']
t = M[(M['축'] == '전체') & M['방식'].isin(base)].set_index('방식').reindex(base)
t = t[['WAPE', 'MASE', '적중률', '배치 인력(명·일)', '부족비용', '잉여비용', '비용']]
t[['부족비용', '잉여비용', '비용']] /= 1e8
t.round(3)

,WAPE,MASE,적중률,배치 인력(명·일),부족비용,잉여비용,비용
방식,,,,,,,
0a 나이브,0.428,1.210,0.400,7764.70,14.770,2.016,16.785
0b 계절 나이브,0.499,1.250,0.501,8892.84,13.264,3.331,16.595
1 직전 7일 평균,0.444,1.224,0.582,8855.35,11.918,2.939,14.857
2 + 서비스 수준 보정,0.444,1.224,0.804,12330.10,6.422,6.777,13.199


## 5. 자주 하는 실수 1 · 5

**실수 1** — 휴관일의 0 을 실제값으로 넣고 채점하면 (원점 52개). 보고서의 0.600 vs 0.480 은 가이드 수치 재현용 원점(2025-07-01 부터 26개) 값이다.

In [6]:
m7 = hv.backtest(panel, hv.fc_mean7, name='직전 7일 평균')
pd.DataFrame({'개방일만 (우리 방식)': hv.summarize(m7).iloc[0][['WAPE']],
              '휴관일 0 포함': hv.summarize(m7, open_only=False).iloc[0][['WAPE']]}).T.round(3)

,WAPE
개방일만 (우리 방식),0.444
휴관일 0 포함,0.607


**실수 5** — `paid == 0` 을 무료 개방일로 보면 (2023~2025 개방일). 창덕궁은 무료일에도 유료를 기록하고, 왕릉은 무료가 아닌 날에도 `paid == 0` 이 많다. `paid` 는 그날이 지나야 아는 값이라 누수이기도 하다.

In [7]:
d = panel[(panel['date'] >= '2023-01-01') & (panel['status'] == 'open')].assign(paid0=lambda x: (x['paid'] == 0))
r = d.groupby('group').apply(lambda x: pd.Series({
    '일치율': (x['paid0'] == (x['is_free_rule'] == 1)).mean(),
    '무료일 중 paid==0': x.loc[x['is_free_rule'] == 1, 'paid0'].mean(),
    '무료 아닌 날 중 paid==0': x.loc[x['is_free_rule'] == 0, 'paid0'].mean()}), include_groups=False)
display(r.round(3))
d[d['is_free_rule'] == 1].groupby('site')['paid0'].mean().loc[['경복궁', '창덕궁', '창경궁', '덕수궁', '종묘']].round(2).rename('무료일 중 paid==0')

,일치율,무료일 중 paid==0,무료 아닌 날 중 paid==0
group,,,
궁,0.963,0.670,0.015
왕릉,0.926,0.997,0.079
종묘,0.981,1.000,0.021


site
경복궁    0.76
창덕궁    0.00
창경궁    1.00
덕수궁    0.92
종묘     1.00
Name: 무료일 중 paid==0, dtype: float64